# CS 839 HW1: Agent memory experiment (no memory, episodic, structured)

Runs the 12 tasks in `data/tasks.json` **in order** (T1 → T12) under three memory conditions and compares them.

- **No memory**: every task starts from scratch. Same requests as `hw1_no_memory.ipynb`.
- **Episodic**: after each task the episode (request, tool calls, answer, verdict) is stored; the 3 most similar past episodes are shown with the next request.
- **Structured**: after each task an LLM writer updates a compact list of rules, procedures, warnings and exceptions (at most 8 records, 150 words), which is shown with every request.

**Feedback is binary.** After each task the memory learns only whether the answer was judged CORRECT or INCORRECT, like a colleague saying "these numbers don't check out". It never sees the reference answer, the grader's failure label, `failure_conditions` or `reference_calls`.

Protocol (from the README): one run = T1 to T12 in order; memory is cleared at the start of each run; 5 runs per condition; conditions interleaved; no memory re-run alongside. Same model (`gpt-5.6-luna`) and tools in every condition, and the memory writer uses the same model.

Code: `utils/agent.py` (the agent loop from `hw1_no_memory.ipynb` plus a memory hook), `utils/memory.py` (the three conditions), `utils/experiment.py` (runs and saved traces), `utils/test_memory.py` (offline checks). Run this notebook from the repo root.

## 1. Setup

The API key is read from a `.env` file in this folder or any parent folder (`OPENAI_API_KEY`, optional `CS839_MODEL`). Otherwise it is entered with `getpass`. Never paste a key into a code cell or commit `.env`.

In [ ]:
%pip install -q -r requirements.txt matplotlib

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from utils.agent import compact_trace
from utils.experiment import CONDITIONS, load_env, load_spec, load_traces, make_client, run_all, spent_usd, trace_cost
from utils.grader import grade_canonical
from utils.memory import EPISODIC_HEADER, STRUCTURED_HEADER, WRITER_INSTRUCTIONS, render_records

env_found = load_env() is not None
MODEL = os.getenv("CS839_MODEL", "gpt-5.6-luna")
if not os.getenv("OPENAI_API_KEY"):
    import getpass
    entered_key = getpass.getpass("Paste your team's API key (hidden): ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key

client = make_client() if os.getenv("OPENAI_API_KEY") else None
print(f"Model: {MODEL} | .env {'found' if env_found else 'not found'}")
print("API client ready." if client else "No key supplied. Saved runs can still be analyzed.")

## 2. Hypotheses (written and committed before any memory-condition run)

Baseline: no memory, 5 runs on Oct 6 (`runs/no_memory_20261006_121523.json`): T1 5/5, T2 3/5, T3 5/5, T4 5/5, T5 0/5, T6 0/5, T7 1/5, T8 5/5, T9 1/5, T10 5/5, T11 4/5, T12 2/5. Almost every miss comes from the default projection EPSG:3857.

**H1 (memory helps).** Structured memory will fix the area and density tasks that come after T2. T5, T6, T7 and T12 will each go from at most 2/5 without memory to at least 4/5, because after T2 the writer will store a rule to use `projection="EPSG:5070"` for area and density. Episodic memory will help less, at most 3/5 on average over T5–T7, because the agent must retrieve the T2 episode and re-derive the rule from a raw trace.

**H2 (memory hurts or has no effect).** Memory will not help on the two exceptions, and can hurt:
- T9 (Alaska–Maine distance): in at least 2 of 5 runs of each memory condition the agent will carry the area rule over to distance and use planar EPSG:5070 (5,801 km instead of the geodesic 5,350.4 km), so neither memory condition will beat no memory on T9 by more than one run.
- T8 (U.S. total): in at least 1 of 5 runs of a memory condition, the "states means the 50 states" lesson from T4–T7 will make the agent leave out DC (341,091,212), while no memory gets 5/5.
- T1, T3, T4, T10 (5/5 without memory): memory will not change accuracy; it will only add input tokens.

## 3. Tasks

`data/tasks.json` is the source of truth (questions, answer formats, reference answers, known mistakes). The order is T1 → T12.

In [ ]:
SPEC = load_spec()
TASKS = {t["id"]: t for t in SPEC["tasks"]}
ORDER = [t["id"] for t in SPEC["tasks"]]

pd.set_option("display.max_colwidth", 140)
pd.DataFrame([{"id": t["id"], "type": t["type"], "rules": ", ".join(t["rules"]), "question": t["question"]}
              for t in SPEC["tasks"]]).set_index("id")

## 4. The memory conditions

| | Episodic | Structured |
|---|---|---|
| **Write** | After each task, store the episode as text: request, every tool call with its arguments and a truncated result, the final answer, and the verdict. No LLM call. | After each task, one writer call (same model) sees the current memory, the episode text, the verdict and the tool descriptions, and returns the full updated memory plus a change log. |
| **Retrieve** | Before each task, TF-IDF cosine (character 4-grams) between the new request and each stored episode; the top 3 are shown. Ties go to the most recent. | The whole memory is shown with every request (it is small). |
| **Update** | None: episodes are never edited. | The writer adds, revises, merges or deletes records. On a conflict it should narrow the old record and add an `exception`. |
| **Forget** | None within a run (at most 11 episodes). | Hard budget of 8 records and 150 words: one re-prompt if over budget, then the trailing records are cut. |

Both memories are cleared at the start of each run. The exact text shown to the agent at each task is saved in the trace (`memory_shown`), and so is the memory after each update (`memory_after`).

In [ ]:
print(EPISODIC_HEADER, "\n")
print(STRUCTURED_HEADER, "\n")
print("WRITER INSTRUCTIONS\n" + WRITER_INSTRUCTIONS.format(max_records=8, max_words=150))

## 5. Offline checks (no API credit)

The grader must pass every reference answer and label every known mistake, and `utils/test_memory.py` checks the memory code with a stub client, including that no forbidden field ever reaches the agent or the writer.

In [ ]:
for t in SPEC["tasks"]:
    assert grade_canonical(t, t["answer"])["correct"], t["id"]
    for fc in t["failure_conditions"]:
        g = grade_canonical(t, fc["answer"])
        assert not g["correct"] and g["label"] == fc["cause"], (t["id"], g)
print("Grader OK on all 12 reference answers and",
      sum(len(t["failure_conditions"]) for t in SPEC["tasks"]), "known mistakes.\n")

out = subprocess.run([sys.executable, "utils/test_memory.py"], capture_output=True, text=True)
print(out.stdout[-2500:] or out.stderr[-2500:])

## 6. Helpers for reading traces

In [ ]:
def show_trace(t, result_chars=200):
    verdict = "CORRECT" if t["correct"] else f"WRONG ({t['label']})"
    print(f"=== {t['condition']} | run {t['run']} | {t['task']} | {verdict} ===")
    if t.get("memory_shown"):
        print("Memory shown to the agent:")
        print("\n".join("  | " + line for line in t["memory_shown"].splitlines()))
    else:
        print("Memory shown to the agent: (none)")
    print(compact_trace(t["question"], t["calls"], t["final_answer"], result_chars))
    changes = (t.get("memory_update") or {}).get("changes") or []
    if changes:
        print("Memory changes after this task:")
        for c in changes:
            print("  -", c)
    print(f"tool calls {t['tool_calls']}, model calls {t['model_calls']}, "
          f"tokens in {t['input_tokens']:,}, out {t['output_tokens']:,}\n")


def sequence(traces, condition, run):
    return sorted((t for t in traces if t["condition"] == condition and t["run"] == run),
                  key=lambda t: ORDER.index(t["task"]))


def show_memory_timeline(traces, condition, run):
    for t in sequence(traces, condition, run):
        mark = "right" if t["correct"] else "WRONG"
        if condition == "structured":
            print(f"--- after {t['task']} ({mark}) ---")
            print(render_records(t["memory_after"]) or "(empty)")
            for c in t["memory_update"].get("changes", []):
                print("    change:", c)
        else:
            stored = ", ".join(f"{e['task']}:{'right' if e['correct'] else 'wrong'}" for e in t["memory_after"])
            print(f"{t['task']} ({mark}) | shown before this task: {t['retrieved'] or '-'} | stored after: {stored}")

## 7. Smoke test: T1–T4 with each memory

A cheap end-to-end check (two short sequences) before the full experiment: are the episodes and the writer's JSON sensible?

In [ ]:
SMOKE_DIR = Path("runs/memory_smoke")
MAIN_DIR = Path("runs/memory_main")
BUDGET_USD = 3.00                      # the course limit is $5 per team; stop well before it
BUDGET_DIRS = (SMOKE_DIR, MAIN_DIR)

if client is not None:
    run_all(client, MODEL, SPEC, SMOKE_DIR, runs=[1], conditions=("episodic", "structured"),
            task_ids=["T1", "T2", "T3", "T4"], workers=2, budget_usd=BUDGET_USD, budget_dirs=BUDGET_DIRS)
smoke = load_traces(SMOKE_DIR)
for condition in ("episodic", "structured"):
    print(f"\n########## {condition} ##########")
    show_memory_timeline(smoke, condition, 1)

## 8. Full experiment: 5 runs × 3 conditions

Each (condition, run) is saved to `runs/memory_main/<condition>_runNN.json` as soon as it finishes, so re-running this cell resumes and never repeats a finished sequence. Sequences are queued run by run (run 1 of all three conditions, then run 2, …), so the conditions are interleaved in time.

In [ ]:
N_RUNS = 5
if client is not None:
    run_all(client, MODEL, SPEC, MAIN_DIR, runs=range(1, N_RUNS + 1), workers=3,
            budget_usd=BUDGET_USD, budget_dirs=BUDGET_DIRS)
print(f"Spent on the memory experiment so far: ${spent_usd(*BUDGET_DIRS):.3f}")

## 9. Stability reruns

To be decided after the main results: close or surprising comparisons are repeated with 5 more runs.

In [ ]:
STABILITY_RUNS = range(6, 11)
STABILITY_CONDITIONS = CONDITIONS
if client is not None:
    run_all(client, MODEL, SPEC, MAIN_DIR, runs=STABILITY_RUNS, conditions=STABILITY_CONDITIONS, workers=3,
            budget_usd=BUDGET_USD, budget_dirs=BUDGET_DIRS)
print(f"Spent on the memory experiment so far: ${spent_usd(*BUDGET_DIRS):.3f}")

## 10. Results

In [ ]:
traces = load_traces(MAIN_DIR)
baseline = json.loads(Path("runs/no_memory_20261006_121523.json").read_text())
for t in baseline:
    t["condition"] = "none (Oct 6)"


def row(t):
    upd = t.get("memory_update") or {}
    return {"condition": t["condition"], "run": t["run"], "task": t["task"], "correct": t["correct"],
            "label": t["label"], "tool_calls": t["tool_calls"], "model_calls": t["model_calls"],
            "input_tokens": t["input_tokens"], "output_tokens": t["output_tokens"],
            "memory_calls": upd.get("memory_calls", 0), "memory_input_tokens": upd.get("memory_input_tokens", 0),
            "memory_output_tokens": upd.get("memory_output_tokens", 0), "cost_usd": trace_cost(t)}


df = pd.DataFrame([row(t) for t in traces])
df_all = pd.concat([df, pd.DataFrame([row(t) for t in baseline])], ignore_index=True)
COLS = ["none (Oct 6)", *CONDITIONS]

counts = df_all.groupby(["task", "condition"])["correct"].agg(["sum", "count"])
table = (counts["sum"].astype(int).astype(str) + "/" + counts["count"].astype(str)).unstack()[COLS].reindex(ORDER)
table.insert(0, "type", [TASKS[i]["type"] for i in ORDER])
overall = df_all.groupby("condition")["correct"].agg(["sum", "count"])
table.loc["all"] = ["", *[f"{overall.loc[c, 'sum']}/{overall.loc[c, 'count']}" for c in COLS]]
Path("results").mkdir(exist_ok=True)
table.to_csv("results/summary_by_task.csv")
table

In [ ]:
per_run = df.groupby(["condition", "run"]).agg(
    correct=("correct", "sum"), tool_calls=("tool_calls", "sum"), model_calls=("model_calls", "sum"),
    task_input_tokens=("input_tokens", "sum"), task_output_tokens=("output_tokens", "sum"),
    memory_calls=("memory_calls", "sum"), memory_input_tokens=("memory_input_tokens", "sum"),
    memory_output_tokens=("memory_output_tokens", "sum"), cost_usd=("cost_usd", "sum"))
cost_table = per_run.groupby("condition").mean().reindex(list(CONDITIONS)).round(3)
cost_table.insert(0, "runs", per_run.groupby("condition").size().reindex(list(CONDITIONS)))
cost_table.to_csv("results/cost_by_condition.csv")
print("Mean per run of 12 tasks. Total spent on this experiment (incl. smoke test): "
      f"${spent_usd(*BUDGET_DIRS):.3f}")
cost_table

In [ ]:
rates = df.pivot_table(index="task", columns="condition", values="correct", aggfunc="mean").reindex(ORDER)[list(CONDITIONS)]
n_runs = df.groupby("condition")["run"].nunique()
colors = {"none": "#9e9e9e", "episodic": "#1f77b4", "structured": "#d62728"}

fig, ax = plt.subplots(figsize=(11, 4))
width = 0.27
for i, c in enumerate(CONDITIONS):
    ax.bar([x + (i - 1) * width for x in range(len(ORDER))], rates[c], width,
           label=f"{c} ({n_runs[c]} runs)", color=colors[c])
for x, tid in enumerate(ORDER):
    if TASKS[tid]["type"] == "exception":
        ax.axvspan(x - 0.5, x + 0.5, color="#fff3cd", zorder=0)
ax.set_xticks(range(len(ORDER)))
ax.set_xticklabels([f"{tid}\n{TASKS[tid]['type']}" for tid in ORDER])
ax.set_ylabel("success rate")
ax.set_ylim(0, 1.05)
ax.set_title("Success across the task sequence (T1 → T12, shaded = exception tasks)")
ax.legend(loc="lower left", ncol=3, frameon=False)
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/success_by_task.png", dpi=200)
plt.show()

In [ ]:
# Wrong answers by cause (grader labels; never shown to the agent or the memory)
wrong = df[~df["correct"]]
pd.crosstab([wrong["task"], wrong["label"]], wrong["condition"]).reindex(columns=list(CONDITIONS), fill_value=0)

## 11. Memory after each task (run 1 of each memory condition)

For episodic memory: which past episodes were shown before each task, and what was stored after it. For structured memory: the full memory after each update, with the writer's change log.

In [ ]:
for condition in ("episodic", "structured"):
    print(f"\n########## {condition}, run 1 ##########")
    show_memory_timeline(traces, condition, 1)

## 12. Final structured memory (every run)

In [ ]:
for run in sorted(df[df["condition"] == "structured"]["run"].unique()):
    last = sequence(traces, "structured", run)[-1]
    acc = sum(t["correct"] for t in sequence(traces, "structured", run))
    print(f"--- structured run {run} ({acc}/12 correct) ---")
    print(render_records(last["memory_after"]), "\n")

## 13. Two trajectories

Candidates are found automatically; the two shown below are discussed in the next cell.

In [ ]:
def find(condition, task, correct):
    return [t for t in traces if t["condition"] == condition and t["task"] == task and t["correct"] == correct]


print("Memory helped (memory right, no memory wrong on the same task):")
for tid in ORDER:
    for c in ("episodic", "structured"):
        right, base_wrong = len(find(c, tid, True)), len(find("none", tid, False))
        if right and base_wrong:
            print(f"  {tid} {c}: right in runs {[t['run'] for t in find(c, tid, True)]}; none wrong {base_wrong}x")
print("\nMemory misled (memory wrong where no memory is mostly right, or a memory-induced label):")
INDUCED = ("Excluded DC", "Planar distance in EPSG:5070", "T8 exception carried over")
for t in traces:
    if t["condition"] != "none" and not t["correct"]:
        base_rate = rates.loc[t["task"], "none"]
        if base_rate >= 0.6 or any(s in (t["label"] or "") for s in INDUCED):
            print(f"  {t['task']} {t['condition']} run {t['run']}: {t['label']} (none: {base_rate:.0%} right)")

In [ ]:
HELPED = ("structured", 1, "T6")
MISLED = ("structured", 1, "T9")
for condition, run, task in (HELPED, MISLED):
    show_trace(next(t for t in traces if (t["condition"], t["run"], t["task"]) == (condition, run, task)))

To be written after the runs.

## 14. Findings

To be written after the runs.